In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

CONST_NAMES = ('const', 'Intercept')


def fmt(x, decimals=3):
    """Число с заданным числом знаков; оценка, которая округляется до нуля, — с тремя значащими цифрами."""
    if pd.isna(x):
        return ''
    text = f'{x:.{decimals}f}'
    if float(text) == 0:
        return f'{x:.3g}' if x != 0 else f'{0:.{decimals}f}'
    return text


def fmt_p(p, decimals=3):
    """P-значение: меньше 0.001 печатается как «<0.001»."""
    return '<0.001' if p < 0.001 else f'{p:.{decimals}f}'


def errors_text(res, case='nom'):
    """Название типа стандартных ошибок подгонки: 'nom' — «неробастные», 'dat' — «по неробастным …»."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    if cov_type == 'nonrobust':
        return {'nom': 'неробастные стандартные ошибки',
                'dat': 'по неробастным стандартным ошибкам'}[case]
    return {'nom': f'робастные стандартные ошибки, {cov_type}',
            'dat': f'по робастным стандартным ошибкам {cov_type}'}[case]


def fit_ols(formula, data, robust=False, **kwargs):
    """МНК-подгонка; robust=True — робастные ошибки HC3. Всегда t- и F-распределения (use_t=True)."""
    cov_type = 'HC3' if robust else 'nonrobust'
    return ols(formula, data=data).fit(cov_type=cov_type, use_t=True, **kwargs)


def show_table(df, index=True, align=None):
    """Таблица pandas в Markdown; числа должны быть заранее превращены в строки через fmt."""
    display(Markdown('\n\n' + df.to_markdown(index=index, colalign=align, disable_numparse=True) + '\n\n'))


def nobs_text(res):
    """Строка «Модель была подогнана по N наблюдениям»."""
    n = int(res.nobs)
    word = 'наблюдению' if n % 10 == 1 and n % 100 != 11 else 'наблюдениям'
    return f'Модель была подогнана по {n} {word}.'


def show_coefs(res, decimals=3, show_nobs=True):
    """Оценки коэффициентов: вертикальная таблица «Переменная | Оценка» и число наблюдений."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    column = 'Оценка' if cov_type == 'nonrobust' else f'Оценка ({cov_type})'
    table = pd.DataFrame({'Переменная': res.params.index,
                          column: [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    if show_nobs:
        text += '\n\n' + nobs_text(res)
    display(Markdown('\n\n' + text + '\n\n'))


def display_answer(text, use_callout=True, theme='note', title='Критическое значение'):
    """Ответ в рамке (callout): theme='note' — число, 'tip' — вывод, 'important' — условие."""
    if use_callout:
        head = f'title="{title}" ' if title else ''
        output_md = f'\n\n::: {{.callout-{theme} {head}icon="true"}}\n{text}\n:::\n\n'
    elif title:
        output_md = f'\n\n**{title}:** {text}\n\n'
    else:
        output_md = f'\n\n{text}\n\n'
    display(Markdown(output_md))


def show_conditions(level, errors=None, hypothesis=None, extra=()):
    """Блок «Условия теста»: уровень значимости (доля или список долей), тип ошибок, гипотеза формулой."""
    levels = level if isinstance(level, (list, tuple)) else [level]
    items = ['уровень значимости: **' + ' и '.join(f'{a * 100:g}%' for a in levels) + '**']
    if errors:
        items.append(f'стандартные ошибки: **{errors}**')
    hypotheses = [hypothesis] if isinstance(hypothesis, str) else list(hypothesis or [])
    if len(hypotheses) == 1:
        items.append(f'гипотеза: {hypotheses[0]}')
    else:
        items += [f'гипотеза {i}: {h}' for i, h in enumerate(hypotheses, start=1)]
    items += list(extra)
    display_answer('\n'.join(f'- {item}' for item in items), theme='important', title='Условия теста')


def signif_stars(p):
    """Коды значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''


SIGNIF_LEGEND = '*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$'


def print_regression_summary(res, decimals=3, show_signif=True):
    """Таблица t-теста: Coef., Std.Err., t-value, p-value, Signif. Заголовок называет тип ошибок."""
    title = f'**Результаты $t$-теста для коэффициентов ({errors_text(res)}):**'
    table = pd.DataFrame({
        'Coef.': [fmt(b, decimals) for b in res.params],
        'Std.Err.': [fmt(s, decimals) for s in res.bse],
        't-value': [fmt(t, decimals) for t in res.tvalues],
        'p-value': [fmt_p(p, decimals) for p in res.pvalues],
    }, index=res.params.index)
    legend = ''
    if show_signif:
        table['Signif.'] = ['' if name in CONST_NAMES else signif_stars(p)
                            for name, p in res.pvalues.items()]
        legend = '\n\n' + SIGNIF_LEGEND
    align = ('left',) + ('right',) * 4 + (('left',) if show_signif else ())
    display(Markdown(f'\n\n{title}\n\n' + table.to_markdown(colalign=align, disable_numparse=True) + legend + '\n\n'))


def print_significant_coeffs(res, sign_level=0.05, theme='note', title=None):
    """Рамка со списком коэффициентов, значимых на уровне sign_level (без константы)."""
    names = [name for name, p in res.pvalues.items() if p < sign_level and name not in CONST_NAMES]
    if title is None:
        title = f'Коэффициенты, значимые на уровне {sign_level * 100:g}% ({errors_text(res, "dat")})'
    text = ', '.join(f'`{name}`' for name in names) if names else 'Значимых коэффициентов не выявлено.'
    display_answer(text, theme=theme, title=title)


def show_models(models, names=None, decimals=3, regressor_order=(), errors_note=None, info=None):
    """Сравнение регрессий (summary_col): строка Dep. Variable, коды значимости, пояснение про скобки.

    info — словарь {название строки: функция от подгонки} для дополнительных строк (N, F и т. п.).
    """
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    table = pd.concat([top, table])
    if errors_note is None:
        errors_note = errors_text(models[0])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_note}.'
    display(Markdown('\n\n' + table.to_markdown(disable_numparse=True) + '\n\n' + note + '\n\n'))


def show_ci(res, level=0.95, decimals=3):
    """Доверительные интервалы: «Переменная | Нижняя граница | Верхняя граница», заголовок над таблицей."""
    ci = res.conf_int(alpha=1 - level)
    table = pd.DataFrame({'Переменная': ci.index,
                          'Нижняя граница': [fmt(x, decimals) for x in ci[0]],
                          'Верхняя граница': [fmt(x, decimals) for x in ci[1]]})
    title = f'**{level * 100:g}%-е доверительные интервалы ({errors_text(res)}):**'
    display(Markdown(f'\n\n{title}\n\n'
                     + table.to_markdown(index=False, colalign=('left', 'right', 'right'), disable_numparse=True) + '\n\n'))


# --- Листок 10: данные и свои помощники ---
from functools import partial
import tabulate
from statsmodels.stats.diagnostic import het_breuschpagan

# to_markdown (tabulate) разбирает строки как числа и срезает нули в конце: «0.0050» -> «0.005».
# Общие помощники этот разбор не отключают, поэтому он отключается здесь для всего листка.
tabulate.tabulate = partial(tabulate.tabulate, disable_numparse=True)

# Загрузка данных
sleep75 = pd.read_csv('../datasets/sleep75.csv')
wage1 = pd.read_csv('../datasets/wage1.csv')
Labour = pd.read_csv('../datasets/Labour.csv')
Electricity = pd.read_csv('../datasets/Electricity.csv')

sign_level = 0.05

# Модели листка (нумерация задач сквозная)
F_SLEEP = 'sleep ~ totwrk + age + I(age**2) + male + smsa + south'
F_WAGE = 'np.log(wage) ~ exper + I(exper**2) + female + married + smsa'
F_OUTPUT = ('np.log(output) ~ np.log(capital) + np.log(labour)'
            ' + I(np.log(capital)**2) + I(np.log(labour)**2)')
F_OUTPUT_WAGE = ('np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage)'
                 ' + I(np.log(capital)**2) + I(np.log(labour)**2) + I(np.log(wage)**2)')
F_COST = 'np.log(cost) ~ np.log(q) + I(np.log(q)**2) + np.log(pf) + np.log(pl) + np.log(pk)'
F_COST_SQ = F_COST + ' + I(np.log(pf)**2) + I(np.log(pl)**2) + I(np.log(pk)**2)'


def bp_test(res):
    """BP-тест, стьюдентизированная версия Коэнкера (nR^2): статистика, степени свободы, P-значение."""
    stat, pval, _, _ = het_breuschpagan(res.resid, res.model.exog)
    return stat, int(res.df_model), pval


def show_bp_conditions():
    show_conditions(sign_level,
                    hypothesis='$H_0$: дисперсия ошибки не зависит от регрессоров модели (гомоскедастичность)',
                    extra=['тест Бройша–Пагана (BP-тест), стьюдентизированная версия Коэнкера: '
                           'статистика $nR^2$, функция `het_breuschpagan`'])


def show_bp(res):
    stat, df, pval = bp_test(res)
    display(Markdown('\n\n**Результат BP-теста:**'))
    show_table(pd.DataFrame({'BP': [fmt(stat, 2)], 'df': [df], 'p-value': [fmt_p(pval)]}),
               index=False, align=('right',) * 3)
    display(Markdown(nobs_text(res) + '\n\n'))


def show_bp_crit(res):
    df = bp_test(res)[1]
    display_answer(f'$\\chi^2_{{crit}}({df}) = {stats.chi2.ppf(1 - sign_level, df):.2f}$')


def show_bp_conclusion(res):
    stat, df, _ = bp_test(res)
    crit = stats.chi2.ppf(1 - sign_level, df)
    if stat > crit:
        text = (f'$BP = {stat:.2f} > {crit:.2f}$ (тест Бройша–Пагана, версия Коэнкера), '
                'гипотеза $H_0$ отвергается: тест указывает на гетероскедастичность.')
    else:
        text = (f'$BP = {stat:.2f} < {crit:.2f}$ (тест Бройша–Пагана, версия Коэнкера), '
                'гипотеза $H_0$ не отвергается: тест указывает на гомоскедастичность.')
    display_answer(text, theme='tip', title='Вывод')


def show_t_conditions():
    show_conditions(sign_level, errors='неробастные и робастные HC3',
                    hypothesis='$H_0:\\beta_j=0$ — для каждого коэффициента отдельно')


def f_test(res, hypothesis):
    """F-тест на подгонке res: статистика, степени свободы, P-значение."""
    test = res.f_test(hypothesis)
    return float(np.squeeze(test.fvalue)), int(test.df_num), int(test.df_denom), float(test.pvalue)


def show_ftest(res, hypothesis):
    F, df1, df2, pval = f_test(res, hypothesis)
    display(Markdown(f'\n\n**Результаты $F$-теста ({errors_text(res)}):**'))
    show_table(pd.DataFrame({'F': [fmt(F, 2)], 'df1': [df1], 'df2': [df2], 'p-value': [fmt_p(pval)]}),
               index=False, align=('right',) * 4)


def F_crit(res, hypothesis):
    _, df1, df2, _ = f_test(res, hypothesis)
    return stats.f.ppf(1 - sign_level, df1, df2), df1, df2


def show_F_crit(res, hypothesis):
    crit, df1, df2 = F_crit(res, hypothesis)
    display_answer(f'$F_{{crit}}({df1}, {df2}) = {crit:.2f}$')


def F_conclusion(res, res_hc, hypothesis):
    """Вывод по робастному F-тесту (HC3) и сравнение с неробастным."""
    crit = F_crit(res_hc, hypothesis)[0]
    F, F_hc = f_test(res, hypothesis)[0], f_test(res_hc, hypothesis)[0]
    verdict = lambda reject: 'отвергается' if reject else 'не отвергается'
    sign = lambda value: '>' if value > crit else '<'
    text = (f'$F = {F_hc:.2f} {sign(F_hc)} {crit:.2f}$ (робастный F-тест, HC3), гипотеза $H_0$ '
            f'{verdict(F_hc > crit)}: влияние {"значимо" if F_hc > crit else "незначимо"}. ')
    if (F > crit) == (F_hc > crit):
        return text + f'Неробастный F-тест даёт тот же вывод ($F = {F:.2f} {sign(F)} {crit:.2f}$).'
    return text + (f'По неробастному F-тесту ($F = {F:.2f} {sign(F)} {crit:.2f}$) гипотеза '
                   f'{verdict(F > crit)} — вывод меняется.')

::: {.callout-important title="Важно"}
- Логарифм натуральный; константа включается в модель по умолчанию.
- Данные лежат в папке `datasets`, описание переменных — в `datasets/data-description.md`.
- Ответы округляются до 2-х десятичных знаков, если в задаче не сказано иное; ответы считаются по неокруглённым оценкам.
- Коэффициенты подписаны как в statsmodels: `Intercept` — константа, `np.log(x)` — $\log x$, `I(x ** 2)` — $x^2$, `I(np.log(x) ** 2)` — $\log^2 x$.
- Тест Бройша–Пагана (BP-тест) — стьюдентизированная версия Коэнкера: статистика $nR^2$, функция `het_breuschpagan`.
- Робастные стандартные ошибки — HC3; P-значения и критические значения — по t- и F-распределениям: `fit(cov_type='HC3', use_t=True)`.
:::

# Диагностика модели: тесты на гетероскедастичность

## sleep equation

In [2]:
res = fit_ols(F_SLEEP, sleep75)

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + I(age**2) + male + smsa + south`

Здесь `sleep` — ночной сон и `totwrk` — работа (минут в неделю), `age` — возраст (лет), `male`, `smsa`, `south` — бинарные переменные.

In [3]:
#| output: asis
show_bp_conditions()



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- гипотеза: $H_0$: дисперсия ошибки не зависит от регрессоров модели (гомоскедастичность)
- тест Бройша–Пагана (BP-тест), стьюдентизированная версия Коэнкера: статистика $nR^2$, функция `het_breuschpagan`
:::



Проведите BP-тест на зависимость дисперсии от регрессоров модели.

In [4]:
#| output: asis
show_bp(res)



**Результат BP-теста:**



|   BP |   df |   p-value |
|-----:|-----:|----------:|
| 8.31 |    6 |     0.216 |



Модель была подогнана по 706 наблюдениям.



Вычислите необходимое критическое значение. **Ответ округлите до 2-х десятичных знаков.**

In [5]:
#| output: asis
show_bp_crit(res)



::: {.callout-note title="Критическое значение" icon="true"}
$\chi^2_{crit}(6) = 12.59$
:::



Какой можно сделать вывод?

In [6]:
#| output: asis
show_bp_conclusion(res)



::: {.callout-tip title="Вывод" icon="true"}
$BP = 8.31 < 12.59$ (тест Бройша–Пагана, версия Коэнкера), гипотеза $H_0$ не отвергается: тест указывает на гомоскедастичность.
:::



## wage equation

In [7]:
res = fit_ols(F_WAGE, wage1)

Для набора данных `wage1` рассмотрим регрессию

`np.log(wage) ~ exper + I(exper**2) + female + married + smsa`

Здесь `wage` — средняя почасовая заработная плата, `exper` — стаж (лет), `female`, `married`, `smsa` — бинарные переменные.

In [8]:
#| output: asis
show_bp_conditions()



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- гипотеза: $H_0$: дисперсия ошибки не зависит от регрессоров модели (гомоскедастичность)
- тест Бройша–Пагана (BP-тест), стьюдентизированная версия Коэнкера: статистика $nR^2$, функция `het_breuschpagan`
:::



Проведите BP-тест на зависимость дисперсии от регрессоров модели.

In [9]:
#| output: asis
show_bp(res)



**Результат BP-теста:**



|   BP |   df |   p-value |
|-----:|-----:|----------:|
| 8.94 |    5 |     0.111 |



Модель была подогнана по 526 наблюдениям.



Вычислите необходимое критическое значение. **Ответ округлите до 2-х десятичных знаков.**

In [10]:
#| output: asis
show_bp_crit(res)



::: {.callout-note title="Критическое значение" icon="true"}
$\chi^2_{crit}(5) = 11.07$
:::



Какой можно сделать вывод?

In [11]:
#| output: asis
show_bp_conclusion(res)



::: {.callout-tip title="Вывод" icon="true"}
$BP = 8.94 < 11.07$ (тест Бройша–Пагана, версия Коэнкера), гипотеза $H_0$ не отвергается: тест указывает на гомоскедастичность.
:::



## output equation #1

In [12]:
res = fit_ols(F_OUTPUT, Labour)

Для набора данных `Labour` рассмотрим регрессию

```
np.log(output) ~ np.log(capital) + np.log(labour) +
    I(np.log(capital)**2) + I(np.log(labour)**2)
```

Здесь `output` — добавленная стоимость и `capital` — основные фонды (млн евро), `labour` — число работников, `wage` — затраты на оплату труда на одного работника (тыс. евро).

In [13]:
#| output: asis
show_bp_conditions()



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- гипотеза: $H_0$: дисперсия ошибки не зависит от регрессоров модели (гомоскедастичность)
- тест Бройша–Пагана (BP-тест), стьюдентизированная версия Коэнкера: статистика $nR^2$, функция `het_breuschpagan`
:::



Проведите BP-тест на зависимость дисперсии от регрессоров модели.

In [14]:
#| output: asis
show_bp(res)



**Результат BP-теста:**



|    BP |   df |   p-value |
|------:|-----:|----------:|
| 44.53 |    4 |    <0.001 |



Модель была подогнана по 569 наблюдениям.



Вычислите необходимое критическое значение. **Ответ округлите до 2-х десятичных знаков.**

In [15]:
#| output: asis
show_bp_crit(res)



::: {.callout-note title="Критическое значение" icon="true"}
$\chi^2_{crit}(4) = 9.49$
:::



Какой можно сделать вывод?

In [16]:
#| output: asis
show_bp_conclusion(res)



::: {.callout-tip title="Вывод" icon="true"}
$BP = 44.53 > 9.49$ (тест Бройша–Пагана, версия Коэнкера), гипотеза $H_0$ отвергается: тест указывает на гетероскедастичность.
:::



## cost equation #1

In [17]:
res = fit_ols(F_COST, Electricity)

Для набора данных `Electricity` рассмотрим регрессию

```
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pf) + np.log(pl) + np.log(pk)
```

Здесь `cost` — общие издержки, `q` — выпуск, `pf` — цена топлива, `pl` — ставка заработной платы, `pk` — индекс цены капитала.

In [18]:
#| output: asis
show_bp_conditions()



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- гипотеза: $H_0$: дисперсия ошибки не зависит от регрессоров модели (гомоскедастичность)
- тест Бройша–Пагана (BP-тест), стьюдентизированная версия Коэнкера: статистика $nR^2$, функция `het_breuschpagan`
:::



Проведите BP-тест на зависимость дисперсии от регрессоров модели.

In [19]:
#| output: asis
show_bp(res)



**Результат BP-теста:**



|    BP |   df |   p-value |
|------:|-----:|----------:|
| 45.08 |    5 |    <0.001 |



Модель была подогнана по 158 наблюдениям.



Вычислите необходимое критическое значение. **Ответ округлите до 2-х десятичных знаков.**

In [20]:
#| output: asis
show_bp_crit(res)



::: {.callout-note title="Критическое значение" icon="true"}
$\chi^2_{crit}(5) = 11.07$
:::



Какой можно сделать вывод?

In [21]:
#| output: asis
show_bp_conclusion(res)



::: {.callout-tip title="Вывод" icon="true"}
$BP = 45.08 > 11.07$ (тест Бройша–Пагана, версия Коэнкера), гипотеза $H_0$ отвергается: тест указывает на гетероскедастичность.
:::



## cost equation #2

In [22]:
res = fit_ols(F_COST_SQ, Electricity)

Для набора данных `Electricity` рассмотрим регрессию

```
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pf) + np.log(pl) + np.log(pk) +
    I(np.log(pf)**2) + I(np.log(pl)**2) + I(np.log(pk)**2)
```

In [23]:
#| output: asis
show_bp_conditions()



::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- гипотеза: $H_0$: дисперсия ошибки не зависит от регрессоров модели (гомоскедастичность)
- тест Бройша–Пагана (BP-тест), стьюдентизированная версия Коэнкера: статистика $nR^2$, функция `het_breuschpagan`
:::



Проведите BP-тест на зависимость дисперсии от регрессоров модели.

In [24]:
#| output: asis
show_bp(res)



**Результат BP-теста:**



|    BP |   df |   p-value |
|------:|-----:|----------:|
| 49.30 |    8 |    <0.001 |



Модель была подогнана по 158 наблюдениям.



Вычислите необходимое критическое значение. **Ответ округлите до 2-х десятичных знаков.**

In [25]:
#| output: asis
show_bp_crit(res)



::: {.callout-note title="Критическое значение" icon="true"}
$\chi^2_{crit}(8) = 15.51$
:::



Какой можно сделать вывод?

In [26]:
#| output: asis
show_bp_conclusion(res)



::: {.callout-tip title="Вывод" icon="true"}
$BP = 49.30 > 15.51$ (тест Бройша–Пагана, версия Коэнкера), гипотеза $H_0$ отвергается: тест указывает на гетероскедастичность.
:::



# Робастный t-тест (HC3)

## output equation #2

In [27]:
res = fit_ols(F_OUTPUT, Labour)
res_hc = fit_ols(F_OUTPUT, Labour, robust=True)

Для набора данных `Labour` рассмотрим регрессию

```
np.log(output) ~ np.log(capital) + np.log(labour) +
    I(np.log(capital)**2) + I(np.log(labour)**2)
```

In [28]:
#| output: asis
print_regression_summary(res, decimals=4)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|                         |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept               | -1.3039 |     0.1886 |   -6.9141 |    <0.001 |           |
| np.log(capital)         |  0.1831 |     0.0166 |   11.0549 |    <0.001 | ***       |
| np.log(labour)          |  0.5153 |     0.0834 |    6.1814 |    <0.001 | ***       |
| I(np.log(capital) ** 2) |  0.0227 |     0.0050 |    4.5181 |    <0.001 | ***       |
| I(np.log(labour) ** 2)  |  0.0203 |     0.0096 |    2.1116 |    0.0352 | **        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



In [29]:
#| output: asis
print_regression_summary(res_hc, decimals=4)
display(Markdown(nobs_text(res) + '\n\n'))
show_t_conditions()



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                         |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept               | -1.3039 |     0.4933 |   -2.6435 |    0.0084 |           |
| np.log(capital)         |  0.1831 |     0.0295 |    6.2148 |    <0.001 | ***       |
| np.log(labour)          |  0.5153 |     0.2064 |    2.4966 |    0.0128 | **        |
| I(np.log(capital) ** 2) |  0.0227 |     0.0083 |    2.7375 |    0.0064 | ***       |
| I(np.log(labour) ** 2)  |  0.0203 |     0.0210 |    0.9654 |    0.3348 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 569 наблюдениям.





::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные и робастные HC3**
- гипотеза: $H_0:\beta_j=0$ — для каждого коэффициента отдельно
:::



Какие коэффициенты значимы?

In [30]:
#| output: asis
print_significant_coeffs(res, sign_level)
print_significant_coeffs(res_hc, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по неробастным стандартным ошибкам)" icon="true"}
`np.log(capital)`, `np.log(labour)`, `I(np.log(capital) ** 2)`, `I(np.log(labour) ** 2)`
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по робастным стандартным ошибкам HC3)" icon="true"}
`np.log(capital)`, `np.log(labour)`, `I(np.log(capital) ** 2)`
:::



## cost equation #3

In [31]:
res = fit_ols(F_COST, Electricity)
res_hc = fit_ols(F_COST, Electricity, robust=True)

Для набора данных `Electricity` рассмотрим регрессию

```
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pf) + np.log(pl) + np.log(pk)
```

In [32]:
#| output: asis
print_regression_summary(res, decimals=4)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|                   |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept         | -6.7387 |     0.7063 |   -9.5412 |    <0.001 |           |
| np.log(q)         |  0.4030 |     0.0316 |   12.7343 |    <0.001 | ***       |
| I(np.log(q) ** 2) |  0.0304 |     0.0022 |   14.0236 |    <0.001 | ***       |
| np.log(pf)        |  0.6847 |     0.0427 |   16.0430 |    <0.001 | ***       |
| np.log(pl)        |  0.1461 |     0.0705 |    2.0729 |    0.0399 | **        |
| np.log(pk)        |  0.1571 |     0.0577 |    2.7214 |    0.0073 | ***       |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



In [33]:
#| output: asis
print_regression_summary(res_hc, decimals=4)
display(Markdown(nobs_text(res) + '\n\n'))
show_t_conditions()



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                   |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept         | -6.7387 |     0.8472 |   -7.9538 |    <0.001 |           |
| np.log(q)         |  0.4030 |     0.0662 |    6.0855 |    <0.001 | ***       |
| I(np.log(q) ** 2) |  0.0304 |     0.0041 |    7.4194 |    <0.001 | ***       |
| np.log(pf)        |  0.6847 |     0.0519 |   13.1882 |    <0.001 | ***       |
| np.log(pl)        |  0.1461 |     0.0854 |    1.7114 |    0.0891 | *         |
| np.log(pk)        |  0.1571 |     0.0623 |    2.5216 |    0.0127 | **        |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 158 наблюдениям.





::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **неробастные и робастные HC3**
- гипотеза: $H_0:\beta_j=0$ — для каждого коэффициента отдельно
:::



Какие коэффициенты значимы?

In [34]:
#| output: asis
print_significant_coeffs(res, sign_level)
print_significant_coeffs(res_hc, sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по неробастным стандартным ошибкам)" icon="true"}
`np.log(q)`, `I(np.log(q) ** 2)`, `np.log(pf)`, `np.log(pl)`, `np.log(pk)`
:::





::: {.callout-note title="Коэффициенты, значимые на уровне 5% (по робастным стандартным ошибкам HC3)" icon="true"}
`np.log(q)`, `I(np.log(q) ** 2)`, `np.log(pf)`, `np.log(pk)`
:::



# Робастный F-тест

## output equation #3

In [35]:
res = fit_ols(F_OUTPUT_WAGE, Labour)
res_hc = fit_ols(F_OUTPUT_WAGE, Labour, robust=True)
hypothesis = 'I(np.log(capital) ** 2) = 0, I(np.log(labour) ** 2) = 0, I(np.log(wage) ** 2) = 0'

Для набора данных `Labour` рассмотрим регрессию

```
np.log(output) ~ np.log(capital) + np.log(labour) + np.log(wage) +
    I(np.log(capital)**2) + I(np.log(labour)**2) + I(np.log(wage)**2)
```

In [36]:
#| output: asis
print_regression_summary(res_hc, decimals=4)
display(Markdown(nobs_text(res) + '\n\n'))
show_conditions(sign_level, errors='робастные, HC3',
                hypothesis='$H_0:\\beta_{\\log^2(capital)}=\\beta_{\\log^2(labour)}=\\beta_{\\log^2(wage)}=0$')



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                         |   Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:------------------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept               | -3.5683 |     1.6513 |   -2.1609 |    0.0311 |           |
| np.log(capital)         |  0.1404 |     0.0297 |    4.7196 |    <0.001 | ***       |
| np.log(labour)          |  0.4716 |     0.1975 |    2.3872 |    0.0173 | **        |
| np.log(wage)            |  0.4762 |     0.8952 |    0.5320 |    0.5949 |           |
| I(np.log(capital) ** 2) |  0.0071 |     0.0085 |    0.8314 |    0.4061 |           |
| I(np.log(labour) ** 2)  |  0.0275 |     0.0200 |    1.3765 |    0.1692 |           |
| I(np.log(wage) ** 2)    |  0.0516 |     0.1257 |    0.4103 |    0.6818 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 569 наблюдениям.





::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **робастные, HC3**
- гипотеза: $H_0:\beta_{\log^2(capital)}=\beta_{\log^2(labour)}=\beta_{\log^2(wage)}=0$
:::



Проверим гипотезу о значимости влияния квадратов переменных.

In [37]:
#| output: asis
show_ftest(res, hypothesis)
show_ftest(res_hc, hypothesis)



**Результаты $F$-теста (неробастные стандартные ошибки):**



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 13.30 |     3 |   562 |    <0.001 |





**Результаты $F$-теста (робастные стандартные ошибки, HC3):**



|    F |   df1 |   df2 |   p-value |
|-----:|------:|------:|----------:|
| 2.06 |     3 |   562 |     0.104 |



Какое критическое значение? **Ответ округлите до 2-х десятичных знаков.**

In [38]:
#| output: asis
show_F_crit(res_hc, hypothesis)



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit}(3, 562) = 2.62$
:::



Значимо ли влияние квадратов переменных?

In [39]:
#| output: asis
display_answer(F_conclusion(res, res_hc, hypothesis), theme='tip', title='Вывод')



::: {.callout-tip title="Вывод" icon="true"}
$F = 2.06 < 2.62$ (робастный F-тест, HC3), гипотеза $H_0$ не отвергается: влияние незначимо. По неробастному F-тесту ($F = 13.30 > 2.62$) гипотеза отвергается — вывод меняется.
:::



## cost equation #4

In [40]:
res = fit_ols(F_COST_SQ, Electricity)
res_hc = fit_ols(F_COST_SQ, Electricity, robust=True)
hypotheses = {name: f'np.log({name}) = 0, I(np.log({name}) ** 2) = 0' for name in ('pl', 'pk', 'pf')}

Для набора данных `Electricity` рассмотрим регрессию

```
np.log(cost) ~ np.log(q) + I(np.log(q)**2) +
    np.log(pf) + np.log(pl) + np.log(pk) +
    I(np.log(pf)**2) + I(np.log(pl)**2) + I(np.log(pk)**2)
```

In [41]:
#| output: asis
print_regression_summary(res, decimals=4)



**Результаты $t$-теста для коэффициентов (неробастные стандартные ошибки):**

|                    |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:-------------------|---------:|-----------:|----------:|----------:|:----------|
| Intercept          | -44.4035 |    23.4317 |   -1.8950 |    0.0600 |           |
| np.log(q)          |   0.3963 |     0.0321 |   12.3329 |    <0.001 | ***       |
| I(np.log(q) ** 2)  |   0.0309 |     0.0022 |   14.0201 |    <0.001 | ***       |
| np.log(pf)         |   0.8988 |     0.6161 |    1.4588 |    0.1467 |           |
| np.log(pl)         |   8.3334 |     5.3296 |    1.5636 |    0.1200 |           |
| np.log(pk)         |   0.4362 |     1.5224 |    0.2865 |    0.7749 |           |
| I(np.log(pf) ** 2) |  -0.0305 |     0.0927 |   -0.3290 |    0.7426 |           |
| I(np.log(pl) ** 2) |  -0.4554 |     0.2967 |   -1.5350 |    0.1269 |           |
| I(np.log(pk) ** 2) |  -0.0360 |     0.1875 |   -0.1923 |    0.8478 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



In [42]:
#| output: asis
print_regression_summary(res_hc, decimals=4)
display(Markdown(nobs_text(res) + '\n\n'))
show_conditions(sign_level, errors='робастные, HC3',
                hypothesis=[f'значимость влияния `{name}`, '
                            f'$H_0:\\beta_{{\\log({name})}}=\\beta_{{\\log^2({name})}}=0$'
                            for name in hypotheses])



**Результаты $t$-теста для коэффициентов (робастные стандартные ошибки, HC3):**

|                    |    Coef. |   Std.Err. |   t-value |   p-value | Signif.   |
|:-------------------|---------:|-----------:|----------:|----------:|:----------|
| Intercept          | -44.4035 |    32.7823 |   -1.3545 |    0.1776 |           |
| np.log(q)          |   0.3963 |     0.0664 |    5.9670 |    <0.001 | ***       |
| I(np.log(q) ** 2)  |   0.0309 |     0.0041 |    7.5071 |    <0.001 | ***       |
| np.log(pf)         |   0.8988 |     1.8426 |    0.4878 |    0.6264 |           |
| np.log(pl)         |   8.3334 |     7.5069 |    1.1101 |    0.2687 |           |
| np.log(pk)         |   0.4362 |     1.6217 |    0.2690 |    0.7883 |           |
| I(np.log(pf) ** 2) |  -0.0305 |     0.2695 |   -0.1132 |    0.9100 |           |
| I(np.log(pl) ** 2) |  -0.4554 |     0.4173 |   -1.0915 |    0.2768 |           |
| I(np.log(pk) ** 2) |  -0.0360 |     0.2003 |   -0.1799 |    0.8575 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 158 наблюдениям.





::: {.callout-important title="Условия теста" icon="true"}
- уровень значимости: **5%**
- стандартные ошибки: **робастные, HC3**
- гипотеза 1: значимость влияния `pl`, $H_0:\beta_{\log(pl)}=\beta_{\log^2(pl)}=0$
- гипотеза 2: значимость влияния `pk`, $H_0:\beta_{\log(pk)}=\beta_{\log^2(pk)}=0$
- гипотеза 3: значимость влияния `pf`, $H_0:\beta_{\log(pf)}=\beta_{\log^2(pf)}=0$
:::



**Гипотеза 1: значимость влияния pl**

Проверим гипотезу о значимости влияния `pl`.

In [43]:
#| output: asis
show_ftest(res, hypotheses['pl'])
show_ftest(res_hc, hypotheses['pl'])



**Результаты $F$-теста (неробастные стандартные ошибки):**



|    F |   df1 |   df2 |   p-value |
|-----:|------:|------:|----------:|
| 3.48 |     2 |   149 |     0.033 |





**Результаты $F$-теста (робастные стандартные ошибки, HC3):**



|    F |   df1 |   df2 |   p-value |
|-----:|------:|------:|----------:|
| 1.98 |     2 |   149 |     0.142 |



**Гипотеза 2: значимость влияния pk**

Проверим гипотезу о значимости влияния `pk`.

In [44]:
#| output: asis
show_ftest(res, hypotheses['pk'])
show_ftest(res_hc, hypotheses['pk'])



**Результаты $F$-теста (неробастные стандартные ошибки):**



|    F |   df1 |   df2 |   p-value |
|-----:|------:|------:|----------:|
| 2.98 |     2 |   149 |     0.054 |





**Результаты $F$-теста (робастные стандартные ошибки, HC3):**



|    F |   df1 |   df2 |   p-value |
|-----:|------:|------:|----------:|
| 2.21 |     2 |   149 |     0.113 |



**Гипотеза 3: значимость влияния pf**

Проверим гипотезу о значимости влияния `pf`.

In [45]:
#| output: asis
show_ftest(res, hypotheses['pf'])
show_ftest(res_hc, hypotheses['pf'])



**Результаты $F$-теста (неробастные стандартные ошибки):**



|      F |   df1 |   df2 |   p-value |
|-------:|------:|------:|----------:|
| 129.37 |     2 |   149 |    <0.001 |





**Результаты $F$-теста (робастные стандартные ошибки, HC3):**



|     F |   df1 |   df2 |   p-value |
|------:|------:|------:|----------:|
| 79.13 |     2 |   149 |    <0.001 |



Какое критическое значение (оно общее для трёх гипотез)? **Ответ округлите до 2-х десятичных знаков.**

In [46]:
#| output: asis
show_F_crit(res_hc, hypotheses['pl'])



::: {.callout-note title="Критическое значение" icon="true"}
$F_{crit}(2, 149) = 3.06$
:::



Значимо ли влияние в каждой из трёх гипотез?

In [47]:
#| output: asis
display_answer('\n'.join(f'- Гипотеза {i} (`{name}`): {F_conclusion(res, res_hc, h)}'
                         for i, (name, h) in enumerate(hypotheses.items(), start=1)),
               theme='tip', title='Вывод')



::: {.callout-tip title="Вывод" icon="true"}
- Гипотеза 1 (`pl`): $F = 1.98 < 3.06$ (робастный F-тест, HC3), гипотеза $H_0$ не отвергается: влияние незначимо. По неробастному F-тесту ($F = 3.48 > 3.06$) гипотеза отвергается — вывод меняется.
- Гипотеза 2 (`pk`): $F = 2.21 < 3.06$ (робастный F-тест, HC3), гипотеза $H_0$ не отвергается: влияние незначимо. Неробастный F-тест даёт тот же вывод ($F = 2.98 < 3.06$).
- Гипотеза 3 (`pf`): $F = 79.13 > 3.06$ (робастный F-тест, HC3), гипотеза $H_0$ отвергается: влияние значимо. Неробастный F-тест даёт тот же вывод ($F = 129.37 > 3.06$).
:::

